# GPT Tokenizer

Following Karpathy's *"Let's build the GPT Tokenizer"* (Zero to Hero, video 8). Implementing Byte-Pair Encoding (BPE) from scratch on raw UTF-8 bytes: start with the 256 byte values as the initial vocabulary, repeatedly find the most-common adjacent pair, merge it into a new token, and repeat until reaching a target vocab size. Then add GPT-style regex pre-tokenization (so BPE only merges within "word-like" chunks, not across them) and compare against the production `tiktoken` library used by GPT-2 and GPT-4.

By the end, the notebook has a complete (toy-scale) tokenizer: `encode(text) -> [token_ids]` and `decode([token_ids]) -> text`, plus a side-by-side comparison of how GPT-2 and GPT-4 tokenize the same input differently.

In [1]:
import regex as re
import tiktoken

In [ ]:
# How Python represents text vs bytes.
#   ord(c)            -> the Unicode codepoint of c (an int)
#   text.encode(...)  -> serializes a string to a sequence of bytes
# UTF-8 is variable-length (1-4 bytes per codepoint). UTF-16 / UTF-32 are wider
# but include byte-order marks (the leading 255 254). GPT uses UTF-8.

# ord() function to see unicode value of given string
print([ord(x) for x in 'Hello World'])

# .encode('utf-x') to see utf encoding
print(list('Hello World'.encode('utf-8')))
print(list('Hello World'.encode('utf-16')))
print(list('Hello World'.encode('utf-32')))

In [3]:
# Large text example
text = 'The FitnessGram Pacer Test is a multistage aerobic capacity test that progressively gets more difficult as it continues. The 20 meter pacer test will begin in 30 seconds. Line up at the start. The running speed starts slowly, but gets faster each minute after you hear this signal. [beep] A single lap should be completed each time you hear this sound. [ding] Remember to run in a straight line, and run as long as possible. The second time you fail to complete a lap before the sound, your test is over. The test will begin on the word start. On your mark, get ready, start.'
tokens = text.encode('utf-8')
tokens = list(map(int, tokens))
print('---' * 25)
print(f'{text=}Length: {len(text)}')
print('---' * 25)
print(f'{tokens=}Length: {len(tokens)}')

---------------------------------------------------------------------------
text='The FitnessGram Pacer Test is a multistage aerobic capacity test that progressively gets more difficult as it continues. The 20 meter pacer test will begin in 30 seconds. Line up at the start. The running speed starts slowly, but gets faster each minute after you hear this signal. [beep] A single lap should be completed each time you hear this sound. [ding] Remember to run in a straight line, and run as long as possible. The second time you fail to complete a lap before the sound, your test is over. The test will begin on the word start. On your mark, get ready, start.'Length: 575
---------------------------------------------------------------------------
tokens=[84, 104, 101, 32, 70, 105, 116, 110, 101, 115, 115, 71, 114, 97, 109, 32, 80, 97, 99, 101, 114, 32, 84, 101, 115, 116, 32, 105, 115, 32, 97, 32, 109, 117, 108, 116, 105, 115, 116, 97, 103, 101, 32, 97, 101, 114, 111, 98, 105, 99, 32, 99, 97, 112,

In [4]:
# Function to find common pairs
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)
# print(sorted(((v, k) for k, v in stats.items()), reverse =True))

In [ ]:
# The most-common adjacent pair in `stats` is our first merge candidate.
# We walk the token list left-to-right, replacing every occurrence of (pair[0], pair[1])
# with a single new token id `idx`. This shortens the sequence by exactly the number of
# matches. After merging, we'd recompute stats and repeat (see the training loop below).
top_pair = max(stats, key=lambda k: stats[k])

def merge(ids, pair, idx):
    # in the list of ints (ids), replace all consecutive occurences of a pair with the new token idx
    newids = []
    i = 0
    while i < len(ids):
        # if we are not at the very last position and the pair matches: replace
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

tokens_new = merge(tokens, top_pair, 256)
print(f'{tokens_new=}Length: {len(tokens_new)}')

In [6]:
# Iterate through the tokens, find common pairs, replace them
text = """According to all known laws of aviation, there is no way a bee should be able to fly. Its wings are too small to get its fat little body off the ground. The bee, of course, flies anyway because bees don't care what humans think is impossible. Yellow, black. Yellow, black. Yellow, black. Yellow, black. Ooh, black and yellow! Let's shake it up a little. Barry! Breakfast is ready! Coming! Hang on a second. Hello? - Barry? - Adam? - Can you believe this is happening? - I can't. I'll pick you up. Looking sharp. Use the stairs. Your father paid good money for those. Sorry. I'm excited. Here's the graduate. We're very proud of you, son. A perfect report card, all B's. Very proud. Ma! I got a thing going here. - You got lint on your fuzz. - Ow! That's me! - Wave to us! We'll be in row 118,000. - Bye! Barry, I told you, stop flying in the house! - Hey, Adam. - Hey, Barry. - Is that fuzz gel? - A little. Special day, graduation. Never thought I'd make it. Three days grade school, three days high school. Those were awkward. Three days college. I'm glad I took a day and hitchhiked around the hive. You did come back different. - Hi, Barry. - Artie, growing a mustache? Looks good. - Hear about Frankie? - Yeah. - You going to the funeral? - No, I'm not going. Everybody knows, sting someone, you die. Don't waste it on a squirrel. Such a hothead. I guess he could have just gotten out of the way. I love this incorporating an amusement park into our day. That's why we don't need vacations. Boy, quite a bit of pomp... under the circumstances. - Well, Adam, today we are men. - We are! - Bee-men. - Amen! Hallelujah! Students, faculty, distinguished bees, please welcome Dean Buzzwell. Welcome, New Hive City graduating class of... ...9:15. That concludes our ceremonies. And begins your career at Honex Industries! Will we pick ourjob today? I heard it's just orientation. Heads up! Here we go. Keep your hands and antennas inside the tram at all times. - Wonder what it'll be like? - A little scary. Welcome to Honex, a division of Honesco and a part of the Hexagon Group. This is it! Wow. Wow. We know that you, as a bee, have worked your whole life to get to the point where you can work for your whole life. Honey begins when our valiant Pollen Jocks bring the nectar to the hive. Our top-secret formula is automatically color-corrected, scent-adjusted and bubble-contoured into this soothing sweet syrup with its distinctive golden glow you know as... Honey! - That girl was hot. - She's my cousin! - She is? - Yes, we're all cousins. - Right. You're right. - At Honex, we constantly strive to improve every aspect of bee existence. These bees are stress-testing a new helmet technology. - What do you think he makes? - Not enough. Here we have our latest advancement, the Krelman. - What does that do? - Catches that little strand of honey that hangs after you pour it. Saves us millions. Can anyone work on the Krelman? Of course. Most bee jobs are small ones. But bees know that every small job, if it's done well, means a lot. But choose carefully because you'll stay in the job you pick for the rest of your life. The same job the rest of your life? I didn't know that. What's the difference? You'll be happy to know that bees, as a species, haven't had one day off in 27 million years. So you'll just work us to death? We'll sure try. Wow! That blew my mind! "What's the difference?" How can you say that? One job forever? That's an insane choice to have to make. I'm relieved. Now we only have to make one decision in life. But, Adam, how could they never have told us that? Why would you question anything? We're bees. We're the most perfectly functioning society on Earth. You ever think maybe things work a little too well here? Like what? Give me one example. I don't know. But you know what I'm talking about. Please clear the gate. Royal Nectar Force on approach. Wait a second. Check it out. - Hey, those are Pollen Jocks! - Wow. I've never seen them this close. They know what it's like outside the hive. Yeah, but some don't come back. - Hey, Jocks! - Hi, Jocks! You guys did great! You're monsters! You're sky freaks! I love it! I love it! - I wonder where they were. - I don't know. Their day's not planned. Outside the hive, flying who knows where, doing who knows what. You can't just decide to be a Pollen Jock. You have to be bred for that. Right. Look. That's more pollen than you and I will see in a lifetime. It's just a status symbol. Bees make too much of it. Perhaps. Unless you're wearing it and the ladies see you wearing it. Those ladies? Aren't they our cousins too? Distant. Distant. Look at these two. - Couple of Hive Harrys. - Let's have fun with them. It must be dangerous being a Pollen Jock. Yeah. Once a bear pinned me against a mushroom! He had a paw on my throat, and with the other, he was slapping me! - Oh, my! - I never thought I'd knock him out. What were you doing during this? Trying to alert the authorities. I can autograph that. A little gusty out there today, wasn't it, comrades? Yeah. Gusty. We're hitting a sunflower patch six miles from here tomorrow. - Six miles, huh? - Barry! A puddle jump for us, but maybe you're not up for it. - Maybe I am. - You are not! We're going 0900 at J-Gate. What do you think, buzzy-boy? Are you bee enough? I might be. It all depends on what 0900 means. Hey, Honex! Dad, you surprised me. You decide what you're interested in? - Well, there's a lot of choices. - But you only get one. Do you ever get bored doing the same job every day? Son, let me tell you about stirring. You grab that stick, and you just move it around, and you stir it around. You get yourself into a rhythm. It's a beautiful thing. You know, Dad, the more I think about it, maybe the honey field just isn't right for me. You were thinking of what, making balloon animals? That's a bad job for a guy with a stinger. Janet, your son's not sure he wants to go into honey! - Barry, you are so funny sometimes. - I'm not trying to be funny. You're not funny! You're going into honey. Our son, the stirrer! - You're gonna be a stirrer? - No one's listening to me! Wait till you see the sticks I have. I could say anything right now. I'm gonna get an ant tattoo! Let's open some honey and celebrate! Maybe I'll pierce my thorax. Shave my antennae. Shack up with a grasshopper. Get a gold tooth and call everybody "dawg"! I'm so proud. - We're starting work today! - Today's the day. Come on! All the good jobs will be gone. Yeah, right. Pollen counting, stunt bee, pouring, stirrer, front desk, hair removal... - Is it still available? - Hang on. Two left! One of them's yours! Congratulations! Step to the side. - What'd you get? - Picking crud out. Stellar! Wow! Oouple of newbies? Yes, sir! Our first day! We are ready! Make your choice. - You want to go first? - No, you go. Oh, my. What's available? Restroom attendant's open, not for the reason you think. - Any chance of getting the Krelman? - Sure, you're on. I'm sorry, the Krelman just closed out. Wax monkey's always open. The Krelman opened up again. What happened? A bee died. Makes an opening. See? He's dead. Another dead one. Deady. Deadified. Two more dead. Dead from the neck up. Dead from the neck down. That's life! Oh, this is so hard! Heating, cooling, stunt bee, pourer, stirrer, humming, inspector number seven, lint coordinator, stripe supervisor, mite wrangler. Barry, what do you think I should... Barry? Barry! All right, we've got the sunflower patch in quadrant nine... What happened to you? Where are you? - I'm going out. - Out? Out where? - Out there. - Oh, no! I have to, before I go to work for the rest of my life. You're gonna die! You're crazy! Hello? Another call coming in. If anyone's feeling brave, there's a Korean deli on 83rd that gets their roses today. Hey, guys. - Look at that. - Isn't that the kid we saw yesterday? Hold it, son, flight deck's restricted. It's OK, Lou. We're gonna take him up. Really? Feeling lucky, are you? Sign here, here. Just initial that. - Thank you. - OK. You got a rain advisory today, and as you all know, bees cannot fly in rain. So be careful. As always, watch your brooms, hockey sticks, dogs, birds, bears and bats. Also, I got a couple of reports of root beer being poured on us. Murphy's in a home because of it, babbling like a cicada! - That's awful. - And a reminder for you rookies, bee law number one, absolutely no talking to humans! All right, launch positions! Buzz, buzz, buzz, buzz! Buzz, buzz, buzz, buzz! Buzz, buzz, buzz, buzz! Black and yellow! Hello! You ready for this, hot shot? Yeah. Yeah, bring it on. Wind, check. - Antennae, check. - Nectar pack, check. - Wings, check. - Stinger, check. Scared out of my shorts, check. OK, ladies, let's move it out! Pound those petunias, you striped stem-suckers! All of you, drain those flowers! Wow! I'm out! I can't believe I'm out! So blue. I feel so fast and free! Box kite! Wow! Flowers! This is Blue Leader. We have roses visual. Bring it around 30 degrees and hold. Roses! 30 degrees, roger. Bringing it around. Stand to the side, kid. It's got a bit of a kick. That is one nectar collector! - Ever see pollination up close? - No, sir. I pick up some pollen here, sprinkle it over here. Maybe a dash over there, a pinch on that one. See that? It's a little bit of magic. That's amazing. Why do we do that? That's pollen power. More pollen, more flowers, more nectar, more honey for us. Cool. I'm picking up a lot of bright yellow. Could be daisies. Don't we need those?  that visual. Wait. One of these flowers seems to be on the move. Say again? You're reporting a moving flower? Affirmative. That was on the line! This is the coolest. What is it? I don't know, but I'm loving this color. It smells good. Not like a flower, but I like it. Yeah, fuzzy. Chemical-y. Careful, guys. It's a little grabby. My sweet lord of bees! Candy-brain, get off there! Problem! - Guys! - This could be bad. Affirmative. Very close."""
tokens = text.encode('utf-8')
tokens = list(map(int, tokens))
print(f'Tokens Length: {len(tokens)}')

Tokens Length: 9979


In [ ]:
# Train the BPE tokenizer: repeatedly find the most-common pair, merge it, record the merge.
# Vocab grows from 256 (raw bytes) up to `vocab_size`. Each merge adds one new token id.
# The `merges` dict maps (p0, p1) -> new_token_id; it's everything we need to encode/decode later.
# Real tokenizers use vocab_size ~ 50k (GPT-2) or ~100k (GPT-4); we use 276 here just for demo.

# Parameters
vocab_size = 276                    # Desired final vocabulary size
num_merges = vocab_size - 256       # Number of merges allowed
ids = list(tokens)                  # Do not destroy original list

merges = {}
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=lambda k: stats[k])
    idx = 256 + i
    
    print(f'{i + 1}:\tMerging {pair} into a new token: {idx}')
    ids = merge(ids, pair, idx)
    merges[pair] = idx

print('\nSTATS ' + '---' * 10)
print(f'Tokens Length: {len(tokens)}')
print(f'ID\'s Length: {len(ids)}')
print(f'Compression Ratio: {len(tokens) / len(ids):.2f}X')    

In [8]:
# Decode a sequence of integers in the range [0, vocab_size]
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode('utf-8', errors='replace')
    return text

print(decode(ids))

According to all known laws of aviation, there is no way a bee should be able to fly. Its wings are too small to get its fat little body off the ground. The bee, of course, flies anyway because bees don't care what humans think is impossible. Yellow, black. Yellow, black. Yellow, black. Yellow, black. Ooh, black and yellow! Let's shake it up a little. Barry! Breakfast is ready! Coming! Hang on a second. Hello? - Barry? - Adam? - Can you believe this is happening? - I can't. I'll pick you up. Looking sharp. Use the stairs. Your father paid good money for those. Sorry. I'm excited. Here's the graduate. We're very proud of you, son. A perfect report card, all B's. Very proud. Ma! I got a thing going here. - You got lint on your fuzz. - Ow! That's me! - Wave to us! We'll be in row 118,000. - Bye! Barry, I told you, stop flying in the house! - Hey, Adam. - Hey, Barry. - Is that fuzz gel? - A little. Special day, graduation. Never thought I'd make it. Three days grade school, three days high

In [ ]:
# Encode a string by greedily applying merges in the order they were learned.
# The subtle bit: at each step we want to apply the EARLIEST learned merge that's still
# applicable. `merges` is an insertion-ordered dict with smaller values for earlier merges,
# so `min(stats, key=lambda p: merges.get(p, +inf))` picks the pair with the lowest merge
# index. The float('inf') default means pairs that were never merged sort last and are
# skipped via the `not in merges` check below.

def encode(text):
    tokens = list(text.encode('utf-8'))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float('inf')))

        if pair not in merges:
            break # Nothing to merge

        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

print(encode('Hello World'))

In [10]:
# Forced splits using regex patterns (GPT Series)
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, 'Hello world how are you'))
example = """
for i in range(1, 101):
    if i % 3 == 0 and i % 5 == 0:
        print("FizzBuzz")
    elif i % 3 == 0:
        print("Fizz")
    elif i % 5 == 0:
        print("Buzz")
    else
        print(i)
"""
print(re.findall(gpt2pat, example))

['Hello', ' world', ' how', ' are', ' you']
['\n', 'for', ' i', ' in', ' range', '(', '1', ',', ' 101', '):', '\n   ', ' if', ' i', ' %', ' 3', ' ==', ' 0', ' and', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'FizzBuzz', '")', '\n   ', ' elif', ' i', ' %', ' 3', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Fizz', '")', '\n   ', ' elif', ' i', ' %', ' 5', ' ==', ' 0', ':', '\n       ', ' print', '("', 'Buzz', '")', '\n   ', ' else', '\n       ', ' print', '(', 'i', ')', '\n']


In [ ]:
# Compare GPT-2 and GPT-4 tokenization on the same input.
# Notice how 'gpt2' encodes the leading whitespace as 5 separate space tokens (220),
# while 'cl100k_base' (GPT-4) merges consecutive whitespace into a single token.
# That's a deliberate design change: GPT-4's tokenizer was retrained on more code,
# where indentation runs are common, so merging whitespace saves a lot of tokens.

enc = tiktoken.get_encoding("gpt2")
print(enc.encode("      Hello World!!!"))

# GPT-4 (merges spaces)
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("      Hello World!!!"))